# 11.11 平均高分掩蓋了哪些弱項？


一個模型在90道常見組合全對，10道稀少組合全錯，總正確率仍有90%。需要處理稀少組合的使用者卻一次也沒成功。平均值沒有說謊，只是把頻繁題佔多數的事實藏進了加權方式。本節用兩種平均說明怎樣把弱項重新看見。

前置是[11.8資料分布影響成績](https://birdhackor.github.io/tiny-perceptron-vlm/11.8.html)。每組應同時保存答對數和總數，而不只存百分比。micro accuracy把全部題合起來算，常見組自然佔較大權重；macro accuracy先算各組正確率，再讓每組同樣重要。兩者回答不同問題，通常可以並排報告。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
groups = {"常見": {"correct": 90, "count": 90}, "稀少": {"correct": 0, "count": 10}}
micro = sum(g["correct"] for g in groups.values()) / sum(g["count"] for g in groups.values())
macro = sum(g["correct"] / g["count"] for g in groups.values()) / len(groups)
for name, group in groups.items():
    print(name, "答對/總數", group["correct"], group["count"])
print("按題平均", micro, "按組平均", macro)

輸入是人工構造的反例，不是真模型成績。`values()`逐組取字典，micro的分子90+0、分母90+10，結果0.9；macro先算90/90=1和0/10=0，再除組數2，結果0.5。輸出原始計數能讓讀者自己重算，發現稀少組確實是0/10。

對視覺問答，可以按顏色、形狀、是否有小物件或未見組合拆組。分組必須在報告中說明，不能只挑有利類別；組別可能重疊，micro也不能把同一道題重複加總。macro給小組同樣權重，但小組若只有一題，結果也很不穩定，因此樣本數仍不可省略。

高macro不自動代表所有組可靠，同樣可以隱藏組內差異。目的不是選出唯一「正確的平均」，而是把總表現與子任務表現放到可檢查的位置。若評估資料本來按真實使用比例取樣，micro對一般使用平均有意義；若關注每類公平表現，macro更容易突出稀少弱項。

例如兩模型總分都0.9，一個常見90/90、稀少0/10；另一個常見81/90、稀少9/10。總答對都90題，卻代表不同使用經驗，後一模型兩組都約0.9。這個對照說明分項不是額外裝飾，而是幫助解釋同分模型該怎麼選擇。若任務必須處理稀少組合，第一模型可能完全不可用；若只報告總分，這種差異看不見。也要把組別定義保存，例如「訓練未見的顏色形狀組合」，而不是事後把所有錯題另起一個方便命名的組。

練習只把稀少組count從10改20，correct仍0。先手算micro=90/110約0.8182，macro仍0.5，再執行核對。稀少組正確率沒有改變，改變的是它在按題平均中的重量。不能從micro下降推論模型學壞了，因為這次我們只改了測試組成。
